# Rodada 3 — respostas distribuídas no tempo

A comparação B01/B02 mantém a mesma amostra. Coeficientes individuais de lags correlacionados podem ser imprecisos; somas e testes conjuntos respondem perguntas diferentes.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0,str(Path.cwd().parent/'testes2'))
from metodos import *
from IPython.display import display
e=Experimento(Path.cwd());d=e.d
# O CSV já traz preço t−1; shifts de 1 e 2 criam t−2 e t−3.
for cultura in ['soja','milho']:
    for lag in [1,2,3]:
        d[f'expo_{cultura}_l{lag}']=d.groupby('id')['expo_'+cultura].shift(lag-1).where(d.ano.eq(d.groupby('id').ano.shift(lag-1)+lag-1))
lagcols=[f'expo_{c}_l{l}' for c in ['soja','milho'] for l in [1,2,3]]

In [2]:
s=d.dropna(subset=['pp_agricultura','pp_pastagem',*lagcols]).copy()

## 1. Preço t−1 na amostra com três lags

In [3]:
for u in ['agricultura','pastagem']:
    e.linear('B01_'+u,'pp_'+u,['expo_soja_l1','expo_milho_l1'],s,familia='robustez_temporal')
display(pd.DataFrame(e.coef)[['id','termo','coef','ic95_inf','ic95_sup','N']])

,id,termo,coef,ic95_inf,ic95_sup,N
0,B01_agricultura,expo_soja_l1,0.028616,-0.020025,0.077258,2539
1,B01_agricultura,expo_milho_l1,0.002733,-0.009494,0.014960,2539
2,B01_pastagem,expo_soja_l1,0.097857,0.022108,0.173606,2539
3,B01_pastagem,expo_milho_l1,0.070227,0.013727,0.126726,2539


## 2. Três lags conjuntamente — dois destinos
## 3. Diferença pastagem−agricultura com os mesmos lags

In [4]:
for ident,y,fam in [('B02_agricultura','pp_agricultura','dinamica'),('B02_pastagem','pp_pastagem','dinamica'),('D02_diferenca','diferenca_pp','diferencas')]:
    r=e.linear(ident,y,lagcols,s,familia=fam)
    for cultura in ['soja','milho']:
        pesos={f'expo_{cultura}_l{l}':1 for l in [1,2,3]}
        e.contraste(ident,'Soma lags1–3 '+cultura,r['x'],r['b'],r['cov']['DK2'],pesos,r['s'],'pp por 1DP persistente preço × 1DP aptidão','cumulativos')
        # Reexpressão econômica do mesmo contraste, não uma hipótese independente.
        pv=d.drop_duplicates('ano')['p_'+cultura];aq=d.drop_duplicates('id')['z_a_'+cultura].quantile([.25,.75]);mult=.1*pv.mean()/pv.std(ddof=0)*(aq.loc[.75]-aq.loc[.25])
        e.contraste(ident,'P75−P25; +10% preço persistente '+cultura,r['x'],r['b'],r['cov']['DK2'],{k:v*mult for k,v in pesos.items()},r['s'],'pp','magnitude_reexpressao')
display(pd.DataFrame(e.mag)[['id','contraste','estimativa','ic95_inf','ic95_sup','p']])
c=pd.DataFrame(e.coef).set_index(['id','termo'])
for term in lagcols:assert np.isclose(c.loc[('D02_diferenca',term),'coef'],c.loc[('B02_pastagem',term),'coef']-c.loc[('B02_agricultura',term),'coef'],atol=1e-9)
e.salvar('dinamica')

,id,contraste,estimativa,ic95_inf,ic95_sup,p
0,B02_agricultura,Soma lags1–3 soja,0.043254,-0.004490,0.090997,7.348007e-02
1,B02_agricultura,P75−P25; +10% preço persistente soja,0.032594,-0.003384,0.068572,7.348007e-02
2,B02_agricultura,Soma lags1–3 milho,0.004091,-0.014494,0.022677,6.518016e-01
3,B02_agricultura,P75−P25; +10% preço persistente milho,0.001053,-0.003730,0.005836,6.518016e-01
4,B02_pastagem,Soma lags1–3 soja,0.162741,0.117011,0.208472,2.803419e-07
5,B02_pastagem,P75−P25; +10% preço persistente soja,0.122636,0.088175,0.157097,2.803419e-07
6,B02_pastagem,Soma lags1–3 milho,0.050238,0.001561,0.098914,4.368875e-02
7,B02_pastagem,P75−P25; +10% preço persistente milho,0.012929,0.000402,0.025457,4.368875e-02
8,D02_diferenca,Soma lags1–3 soja,0.119487,0.068009,0.170966,9.026689e-05
9,D02_diferenca,P75−P25; +10% preço persistente soja,0.090041,0.051249,0.128834,9.026689e-05
